# 01 - Environment Setup & Dataset Preparation
**Cost-Effective Domain RAG with Fine-tuned Matryoshka Embeddings**

### Objectives:
1. verify Colab GPU runtime  (NVIDIA T4 16GB recommend).
2. Mount Google Drive for persistent checkpoint and data storage.
3. Install pinned dependencies.
4. Download and inspect the **BEIR SciFact** dataset using `src.data_loader`.

In [1]:
import os

# 1. Clone only if not already cloned
if not os.path.exists("matryoshka-domain-rag") and not os.path.exists("src"):
    !git clone https://github.com/premsaipusapati-debug/matryoshka-domain-rag.git
    %cd matryoshka-domain-rag
elif os.path.exists("matryoshka-domain-rag"):
    %cd matryoshka-domain-rag

# 2. Pull any recent changes from GitHub
!git pull

# 3. Install requirements
!pip install -q -r requirements.txt


Cloning into 'matryoshka-domain-rag'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 15 (delta 0), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 373.17 KiB | 1.49 MiB/s, done.
/content/matryoshka-domain-rag
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 65.3 MB/s eta 0:00:00


In [2]:
# Check GPU allocation
!nvidia-smi

# Mount Google Drive for persistent storage
from google.colab import drive
import os

drive.mount('/content/drive')
drive_save_dir = "/content/MyDrive/Matryoshka-RAG"
os.makedirs(drive_save_dir, exist_ok=True)
print(f"Persistaent storage confiemed at: {drive_save_dir}")

Wed Sep 23 17:48:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Install project dependencies
!pip install -q "sentence-transformer>=2.5.0" "transformers>=4.38.0" " datasets>=2.18.0" "faiss-cpu>=1.7.4" pyyaml

import sys
import os

# Add aprent directory to sys.pth so 'src' is importable
current_dir = os.getcwd()
if os.path.basename(current_dir) == "notebooks":
  sys.path.append(os.path.abspath(".."))
else:
  sys.path.append(current_dir)

import src
print(f"src package loaded successfully! Version: {src.__version__}")

ERROR: Could not find a version that satisfies the requirement sentence-transformer>=2.5.0 (from versions: none)
ERROR: No matching distribution found for sentence-transformer>=2.5.0
src package loaded successfully! Version: 1.0.0


In [4]:
from src.data_loader import load_scifact_raw, parse_corpus, parse_queries

raw_data = load_scifact_raw("mteb/scifact")

corpus = parse_corpus(raw_data["corpus"])
queries = parse_queries(raw_data["queries"])

print(f"\n--- SciFact Dataset Statistics ---")
print(f"Total Corpus Documents: {len(corpus):,}")
print(f"Total Queries (Claims): {len(queries):,}")
print(f"Train Qrels: {len(raw_data['qrels_train']):,}")
print(f"Test Qrels:  {len(raw_data['qrels_test']):,}")

README.md:   0%|          | 0.00/8.25k [00:00<?, ?B/s]

corpus.jsonl:   0%|          | 0.00/8.02M [00:00<?, ?B/s]

Generating corpus split:   0%|          | 0/5183 [00:00<?, ? examples/s]

queries.jsonl:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating queries split:   0%|          | 0/1109 [00:00<?, ? examples/s]

train.jsonl:   0%|          | 0.00/54.0k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/919 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/339 [00:00<?, ? examples/s]


--- SciFact Dataset Statistics ---
Total Corpus Documents: 5,183
Total Queries (Claims): 1,109
Train Qrels: 919
Test Qrels:  339


In [5]:
from src.data_loader import create_training_data, create_training_dataloader

train_examples = create_training_data(raw_data)
train_loader = create_training_dataloader(train_examples, batch_size=32)

print(f"Total Training Examples: {len(train_examples):,}")
print(f"Total Training Batches (batch_size=32): {len(train_loader):,}")

# Display a sample positive pair
sample = train_examples[0]
print("\n--- Sample Training Pair ---")
print(f"[Query]:\n{sample.texts[0]}\n")
print(f"[Positive Document]:\n{sample.texts[1][:250]}...")

Total Training Examples: 919
Total Training Batches (batch_size=32): 28

--- Sample Training Pair ---
[Query]:
0-dimensional biomaterials lack inductive properties.

[Positive Document]:
New opportunities: the use of nanotechnologies to manipulate and track stem cells. Nanotechnologies are emerging platforms that could be useful in measuring, understanding, and manipulating stem cells. Examples include magnetic nanoparticles and quan...
